# Convolutional neural networks: a hands-on implementation

**Track:** Neural networks · **Guide:** [`020-cnn.md`](../guides/concepts/020-cnn.md)

A CNN applies small shared filters across an image to detect local patterns such as edges and textures.

> **Intuition:** A stencil scans every part of a picture with the same pattern detector.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Convolutional neural networks]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Classify manufacturing images while checking that the model attends to the defect, not the background.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Create tiny labeled images

Synthetic 5-by-5 images contain a vertical scratch, horizontal scratch, or no defect. They make convolution visible without downloads.


In [ ]:
import numpy as np


def make_image(kind):
    image = np.zeros((5, 5), dtype=float)
    if kind == "vertical":
        image[:, 2] = 1
    elif kind == "horizontal":
        image[2, :] = 1
    elif kind != "clean":
        raise ValueError(f"unknown kind: {kind}")
    return image

labels = ["vertical", "horizontal", "clean"]
images = np.stack([make_image(label) for label in labels])
print("batch shape:", images.shape)
print(images[0])

## 2. Convolve with a shared edge filter

Slide the same small kernel across each image: weight sharing detects local patterns wherever they occur.


In [ ]:
vertical_kernel = np.array([[-1, 2, -1], [-1, 2, -1], [-1, 2, -1]], dtype=float)

def convolve2d(image, kernel):
    height, width = image.shape
    kh, kw = kernel.shape
    output = np.zeros((height - kh + 1, width - kw + 1))
    for row in range(output.shape[0]):
        for col in range(output.shape[1]):
            output[row, col] = np.sum(image[row:row + kh, col:col + kw] * kernel)
    return output

feature_maps = [convolve2d(image, vertical_kernel) for image in images]
print("vertical responses:", [round(float(fmap.max()), 1) for fmap in feature_maps])

## 3. Pool and classify by strongest evidence

Global max pooling compresses each feature map. This hand-built detector illustrates a CNN component, not a trained deep network.


In [ ]:
scores = [float(fmap.max()) for fmap in feature_maps]
predictions = [
    "vertical" if score > 0 else "not-vertical"
    for score in scores
]
for label, score, prediction in zip(labels, scores, predictions):
    print(f"true={label:10s} feature={score:4.1f} detector={prediction}")

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/020-cnn.md) and the
[learning roadmap](../guides/ROADMAP.md).
